# Pip voice-control models — "hey pip" + "yes"

Trains **both** microWakeWord models the firmware embeds, in one run:

| slot | phrase(s) | armed | firmware file |
|---|---|---|---|
| `wake` | hey pip | always | `voice_model_wake.c` |
| `confirm` | yes / yeah / yep | answer windows only | `voice_model_confirm.c` |

**How to run**
1. Runtime → Change runtime type → **GPU**. An **A100 (Colab Pro, + High-RAM)**
   finishes both models in ~1.5–2 h. A free **T4** works but is slow and can
   OOM during validation — set `LOW_RESOURCE = True` below and expect ~3–4 h
   *per word* (train one word per session by editing `TRAIN_SLOTS`).
2. Runtime → **Run all**. Everything else is automatic.
3. The last cell downloads `pip_voice_models.zip` (also pushed to Drive if
   the optional mount cell succeeded). Back on the Mac:
   `unzip` into `tools/wakeword/models/`, then `./install_models.sh`.

Pipeline: piper-sample-generator synthesizes thousands of voiced samples per
phrase → augmentation (room reverb, background noise, EQ, pitch) →
spectrogram features → mixednet streaming model → int8 quantized TFLite.
Negative data is kahrendt's pre-generated feature sets plus per-word
*confusables* (phonetic near-misses that must NOT fire — including "no"
for the confirm model). Based on OHF-Voice/micro-wake-word and the
community microwakeword-trainer fixes (MIT).


In [ ]:
# ================== CONFIG — the only cell to edit ==================
LOW_RESOURCE = False   # True on a free T4: fewer samples, smaller batches
TRAIN_SLOTS = ["confirm"]   # wake is trained + committed (models/
                            # hey_pip.tflite) - set ["wake", "confirm"]
                            # to retrain both

SLOTS = {
    "wake": {
        "output_name": "hey_pip",
        "wake_word": "Hey Pip",
        # espeak-ng -q --ipa "hey pip" -> transcribed US/UK variants
        "positives": [("h\u02c8e\u026a p\u02c8\u026ap", 1.0)],
        "confusables": [
            # generic hey-X and other assistants (must not steal them)
            "hey there", "hey you", "hey now", "hey siri", "hey google",
            "okay google", "hey alexa", "okay nabu", "hey jarvis",
            # phonetic neighbors of "pip"
            "hey pete", "hey pippa", "hey philip", "hey rip", "hey dip",
            "hey tip", "hey pig", "hey bib", "hey pap", "hey pop",
            # bare name and mash-ups
            "pip", "a peep", "hiccup",
        ],
    },
    "confirm": {
        "output_name": "yes",
        "wake_word": "Yes",
        # shorter word, armed only in answer windows: half-length
        # schedule (the wake run showed metrics plateau well before the
        # full 45k)
        "training_steps": [12000, 8000],
        # accept the natural family: yes, yeah, yep
        "positives": [("j\u02c8\u025bs", 0.6),
                      ("j\u02c8\u025b\u0259", 0.2),   # yeah
                      ("j\u02c8\u025bp", 0.2)],        # yep
        "confusables": [
            # the one that must NEVER read as yes
            "no", "no no", "nope", "not yet", "oh no",
            # phonetic neighbors
            "less", "mess", "guess", "bless", "chess", "yet",
            "yesterday", "excess", "us",
        ],
    },
}

SAMPLES_POSITIVE = 6000 if LOW_RESOURCE else 12000   # per slot, split by weight
SAMPLES_PER_CONFUSABLE = 300 if LOW_RESOURCE else 600
PIPER_BATCH = 128 if LOW_RESOURCE else 256
TRAINING_STEPS = [15000, 10000] if LOW_RESOURCE else [25000, 20000]
BATCH_SIZE = 128 if LOW_RESOURCE else 256

DRIVE_FOLDER = "pip_wakeword_training"   # results copied here if Drive mounts
print("Slots to train:", TRAIN_SLOTS, "| LOW_RESOURCE =", LOW_RESOURCE)


In [ ]:
# === GPU check ===
import subprocess
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("No GPU! Runtime -> Change runtime type -> GPU, then rerun.")
gpu = r.stdout.strip()
print("GPU:", gpu)
if "T4" in gpu and not LOW_RESOURCE:
    print("WARNING: T4 detected - strongly consider LOW_RESOURCE = True "
          "(validation can exhaust system RAM on T4 runtimes)")
import shutil
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Disk free: {free_gb:.0f} GB")
if free_gb < 90:
    print("WARNING: small disk (T4/L4-class VM). The pipeline fits, but "
          "prefer an A100 runtime (~235 GB) for headroom.")


In [ ]:
# === Mount Drive (optional - results survive a dead runtime) ===
DRIVE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    DRIVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Results will also be copied to", DRIVE_DIR)
except Exception as e:
    print("Drive not mounted (fine - the last cell downloads a zip):", e)


In [ ]:
# === Install microWakeWord (kernel-restart-free) ===
# Fixes lifted from the community microwakeword-trainer notebook:
#  1. no `pip install -e` (needs kernel restart) and no plain install
#     (setup.py misses the audio/ subpackage) -> deps + sys.path.insert
#  2. train.py calls .numpy() on values newer TF already returns as numpy
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert', 'huggingface_hub',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = f'{MWW_DIR}/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
                 r'(\1.numpy() if hasattr(\1, "numpy") else \1)', src)
if patched != src:
    open(fp, 'w').write(patched)
    print('Patched .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword imports clean')


In [ ]:
# === Piper sample generator (TTS positives) ===
import glob, os, shutil, subprocess, sys, urllib.request

PIPER_REPO_DIR = '/content/piper'
PSG_DIR = '/content/piper-sample-generator'

subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'pip', 'setuptools', 'wheel', 'cython'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'piper-tts', 'piper-sample-generator'], check=True)

if not os.path.exists(PIPER_REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/rhasspy/piper', PIPER_REPO_DIR], check=True)
if not os.path.exists(PSG_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/rhasspy/piper-sample-generator', PSG_DIR],
                   check=True)

# build the monotonic_align C extension where piper_train expects it
PIPER_PY = f'{PIPER_REPO_DIR}/src/python'
MA = f'{PIPER_PY}/piper_train/vits/monotonic_align'
shutil.rmtree(f'{PIPER_PY}/build', ignore_errors=True)
shutil.rmtree(f'{MA}/monotonic_align', ignore_errors=True)
shutil.rmtree(f'{MA}/piper_train', ignore_errors=True)
os.makedirs(f'{MA}/monotonic_align', exist_ok=True)
os.makedirs(f'{MA}/piper_train/vits/monotonic_align', exist_ok=True)
open(f'{MA}/monotonic_align/__init__.py', 'a').close()
subprocess.run(f'cd {MA} && {sys.executable} setup.py build_ext --inplace',
               shell=True, check=True)
built = next(iter(glob.glob(f'{MA}/piper_train/vits/monotonic_align/core.*')), None)
assert built, 'monotonic_align build failed'
shutil.copy2(built, f'{MA}/monotonic_align/')

for p in (PIPER_PY, PSG_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
os.makedirs('models', exist_ok=True)
if not os.path.exists(MODEL_PATH):
    print('Downloading libritts_r generator (~75 MB)...')
    urllib.request.urlretrieve(
        'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
        MODEL_PATH)
    urllib.request.urlretrieve(
        'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json',
        MODEL_PATH + '.json')
print('Piper ready')


In [ ]:
# === Shared negative datasets + augmentation corpora (slow, ~6 GB) ===
import os, subprocess
from pathlib import Path

# negative feature sets: kahrendt/microwakeword publishes them as zips
os.makedirs('/content/negative_datasets', exist_ok=True)
for z in ('speech', 'dinner_party', 'dinner_party_eval', 'no_speech'):
    d = f'/content/negative_datasets/{z}'
    if os.path.exists(d) and os.listdir(d):
        print(z, 'ready'); continue
    print(f'downloading {z}.zip ...')
    subprocess.run(
        'cd /content/negative_datasets && '
        f'wget -q https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/{z}.zip && '
        f'unzip -q -o {z}.zip && rm {z}.zip', shell=True, check=True)
    assert os.path.exists(d) and os.listdir(d), f'{z}.zip: unexpected layout'
    print(z, 'ok')

# room impulse responses for reverb augmentation
if not os.path.exists('mit_rirs') or not os.listdir('mit_rirs'):
    print('Downloading room impulse responses...')
    subprocess.run('mkdir -p mit_rirs && cd mit_rirs && '
                   'wget -q https://www.openslr.org/resources/28/rirs_noises.zip && '
                   'unzip -q rirs_noises.zip && rm rirs_noises.zip',
                   shell=True, check=True)

# FMA music background: convert mchl914/fma_xsmall mp3s with ffmpeg.
# (Deliberately NOT via the `datasets` library - its audio decoding API
# drifts; ffmpeg is always present on Colab and boring.)
if not (os.path.exists('fma_16k') and len(os.listdir('fma_16k')) > 100):
    print('downloading FMA xsmall (~180 MB)...')
    subprocess.run(
        'mkdir -p fma && cd fma && '
        'wget -q https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip && '
        'unzip -q -o fma_xs.zip && rm fma_xs.zip', shell=True, check=True)
    os.makedirs('fma_16k', exist_ok=True)
    mp3s = sorted(str(p) for p in Path('fma').glob('**/*.mp3'))
    print(f'converting {len(mp3s)} mp3s to 16 kHz wav...')
    done = fail = 0
    for i, mp3 in enumerate(mp3s):
        out = f'fma_16k/{i:05d}.wav'
        if os.path.exists(out):
            done += 1; continue
        r = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', mp3,
                            '-ac', '1', '-ar', '16000', out])
        if r.returncode == 0: done += 1
        else: fail += 1        # FMA ships a few corrupt mp3s; skip them
    print(f'fma_16k: {done} clips ({fail} skipped)')
    assert done > 100, 'FMA conversion mostly failed'
    subprocess.run('rm -rf fma', shell=True)   # mp3 sources no longer needed
print('Negative datasets + backgrounds ready '
      f'({len(os.listdir("fma_16k"))} background clips)')


In [ ]:
# === Per-slot pipeline (generate -> augment -> features -> train -> export) ===
import os, sys, shutil, subprocess, traceback, yaml
from pathlib import Path
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

MODEL_PATH = 'models/en_US-libritts_r-medium.pt'


def run_piper(phrase, n, out_dir, phoneme=False):
    # pip package (>= 3.x): module invocation, no top-level script. The
    # .pt generator is a pickle referencing piper_train.vits, which the
    # wheel does NOT ship - the piper clone provides it, and PYTHONPATH
    # must carry it into the subprocess (notebook sys.path does not).
    env = os.environ.copy()
    env['PYTHONPATH'] = '/content/piper/src/python:' + env.get('PYTHONPATH', '')
    cmd = [sys.executable, '-m', 'piper_sample_generator', phrase,
           '--model', MODEL_PATH, '--max-samples', str(n),
           '--batch-size', str(PIPER_BATCH),
           '--noise-scales', '0.5', '--noise-scale-ws', '0.6',
           '--output-dir', out_dir]
    if phoneme:
        cmd.append('--phoneme-input')
    subprocess.run(cmd, check=True, env=env)


def generate_positives(slot):
    cfg = SLOTS[slot]
    out = f'generated_samples_{slot}'
    os.makedirs(out, exist_ok=True)
    if len(os.listdir(out)) > SAMPLES_POSITIVE * 0.9:
        print(f'{slot}: positives cached'); return out
    for i, (ipa, weight) in enumerate(cfg['positives']):
        n = int(SAMPLES_POSITIVE * weight)
        tmp = f'/tmp/pos_{slot}_{i}'
        os.makedirs(tmp, exist_ok=True)
        print(f'{slot}: {n} samples for /{ipa}/ ...')
        run_piper(ipa, n, tmp, phoneme=True)
        for f in os.listdir(tmp):
            if f.endswith('.wav'):
                os.rename(f'{tmp}/{f}', f'{out}/p{i}_{f}')
    print(f'{slot}: {len(os.listdir(out))} positive samples')
    return out


def generate_confusables(slot):
    cfg = SLOTS[slot]
    out = f'confusable_negatives_{slot}'
    os.makedirs(out, exist_ok=True)
    for phrase in cfg['confusables']:
        safe = phrase.replace(' ', '_').replace("'", '')
        if len(list(Path(out).glob(f'{safe}_*.wav'))) >= SAMPLES_PER_CONFUSABLE:
            continue
        tmp = f'/tmp/conf_{slot}_{safe}'
        os.makedirs(tmp, exist_ok=True)
        print(f'{slot}: {SAMPLES_PER_CONFUSABLE} confusables for {phrase!r}...')
        run_piper(phrase, SAMPLES_PER_CONFUSABLE, tmp)
        for f in os.listdir(tmp):
            if f.endswith('.wav'):
                os.rename(f'{tmp}/{f}', f'{out}/{safe}_{f}')
    print(f'{slot}: {len(os.listdir(out))} confusable samples')
    return out


def make_augmenter():
    backgrounds = [d for d in ('fma_16k', 'audioset_16k')
                   if os.path.exists(d) and os.listdir(d)]
    return Augmentation(
        augmentation_duration_s=3.2,
        augmentation_probabilities={
            'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
            'PitchShift': 0.15, 'BandStopFilter': 0.10,
            'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
            'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
        },
        impulse_paths=['mit_rirs'],
        background_paths=backgrounds,
        background_min_snr_db=-5, background_max_snr_db=20,
        min_jitter_s=0.10, max_jitter_s=0.50)


SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 2, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1},
}


def make_features(sample_dir, feature_dir):
    if not (os.path.exists(sample_dir) and os.listdir(sample_dir)):
        return False
    clips = Clips(input_directory=sample_dir, file_pattern='*.wav',
                  max_clip_duration_s=None, remove_silence=True,
                  random_split_seed=42, split_count=0.1)
    augmenter = make_augmenter()
    for split, cfg in SPLIT_CONFIG.items():
        mmap = f'{feature_dir}/{split}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            print(f'{feature_dir}/{split}: cached'); continue
        shutil.rmtree(mmap, ignore_errors=True)
        os.makedirs(f'{feature_dir}/{split}', exist_ok=True)
        print(f'{feature_dir}/{split} (rep={cfg["repetition"]})...')
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                   slide_frames=cfg['slide_frames'], step_ms=10)
        try:
            RaggedMmap.from_generator(
                out_dir=mmap, batch_size=200, verbose=True,
                sample_generator=sg.spectrogram_generator(
                    split=cfg['split_name'], repeat=cfg['repetition']))
        except Exception:
            traceback.print_exc()
            shutil.rmtree(mmap, ignore_errors=True)
            raise
    return True


def write_config(slot, has_confusables, has_real):
    name = SLOTS[slot]['output_name']
    features = [
        dict(features_dir=f'features_{slot}_pos', sampling_weight=8.0,
             penalty_weight=2.0, truth=True,
             truncation_strategy='truncate_start', type='mmap'),
        dict(features_dir='/content/negative_datasets/speech', sampling_weight=10.0,
             penalty_weight=2.5, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='/content/negative_datasets/dinner_party', sampling_weight=15.0,
             penalty_weight=3.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='/content/negative_datasets/no_speech', sampling_weight=5.0,
             penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='/content/negative_datasets/dinner_party_eval', sampling_weight=0.0,
             penalty_weight=1.0, truth=False, truncation_strategy='split', type='mmap'),
    ]
    if has_confusables:
        features.append(dict(features_dir=f'features_{slot}_conf', sampling_weight=8.0,
                             penalty_weight=5.0, truth=False,
                             truncation_strategy='random', type='mmap'))
    if has_real:
        features.append(dict(features_dir=f'features_{slot}_real', sampling_weight=8.0,
                             penalty_weight=2.0, truth=True,
                             truncation_strategy='truncate_start', type='mmap'))
    steps = SLOTS[slot].get('training_steps', TRAINING_STEPS)
    config = {
        'window_step_ms': 10, 'train_dir': f'trained_models/{name}',
        'features': features,
        'training_steps': steps,
        'positive_class_weight': [2] * len(steps),
        'negative_class_weight': [40, 50][:len(steps)],
        'learning_rates': [0.001, 0.0001][:len(steps)],
        'batch_size': BATCH_SIZE,
        'time_mask_max_size': [5] * len(steps),
        'time_mask_count': [1] * len(steps),
        'freq_mask_max_size': [3] * len(steps),
        'freq_mask_count': [1] * len(steps),
        'eval_step_interval': 500, 'clip_duration_ms': 1500,
        'target_minimization': 0.4,
        'minimization_metric': 'ambient_false_positives_per_hour',
        'maximization_metric': 'average_viable_recall',
    }
    os.makedirs(f'trained_models/{name}', exist_ok=True)
    path = f'training_parameters_{slot}.yaml'
    with open(path, 'w') as f:
        yaml.dump(config, f)
    return path


def train(slot):
    name = SLOTS[slot]['output_name']
    shutil.rmtree(f'trained_models/{name}', ignore_errors=True)
    env = os.environ.copy()
    env['PYTHONPATH'] = '/content/microWakeWord:' + env.get('PYTHONPATH', '')
    env['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'
    cmd = [sys.executable, '-m', 'microwakeword.model_train_eval',
           '--training_config', f'training_parameters_{slot}.yaml',
           '--train', '1', '--restore_checkpoint', '0',
           '--test_tflite_streaming_quantized', '1',
           '--use_weights', 'best_weights',
           'mixednet',
           '--pointwise_filters', '64,64,64,64',
           '--repeat_in_block', '1, 1, 1, 1',
           '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
           '--residual_connection', '0,0,0,0',
           '--first_conv_filters', '32',
           '--first_conv_kernel_size', '5',
           '--stride', '3']
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    assert proc.returncode == 0, f'{slot}: training failed, see log above'
    src = (f'trained_models/{name}/tflite_stream_state_internal_quant/'
           'stream_state_internal_quant.tflite')
    assert os.path.exists(src), f'{slot}: no tflite at {src}'
    shutil.copy2(src, f'{name}.tflite')
    print(f'{slot}: wrote {name}.tflite '
          f'({os.path.getsize(f"{name}.tflite")/1024:.1f} KB)')
    if DRIVE_DIR:
        shutil.copy2(f'{name}.tflite', DRIVE_DIR)


def disk_report(tag):
    free = shutil.disk_usage('/content').free / 1e9
    print(f'[disk] {tag}: {free:.0f} GB free')
    if free < 8:
        raise RuntimeError('less than 8 GB free - about to fill the disk')


def run_slot(slot):
    disk_report(f'{slot} start')
    pos = generate_positives(slot)
    conf = generate_confusables(slot)
    disk_report(f'{slot} samples done')
    make_features(pos, f'features_{slot}_pos')
    has_conf = make_features(conf, f'features_{slot}_conf')
    # optional real recordings: upload wavs to real_recordings_<slot>/
    has_real = make_features(f'real_recordings_{slot}', f'features_{slot}_real')
    # source wavs are baked into the feature mmaps now - reclaim the disk
    shutil.rmtree(pos, ignore_errors=True)
    shutil.rmtree(conf, ignore_errors=True)
    disk_report(f'{slot} features done')
    write_config(slot, has_conf, bool(has_real))
    train(slot)
    # the tflite is exported + on Drive; drop this slot's bulky leftovers
    for d in (f'features_{slot}_pos', f'features_{slot}_conf',
              f'features_{slot}_real', f'trained_models/{SLOTS[slot]["output_name"]}'):
        shutil.rmtree(d, ignore_errors=True)
    disk_report(f'{slot} finished')

print('Pipeline ready')


In [ ]:
# === Train slot: wake ("hey pip") ===
if "wake" in TRAIN_SLOTS:
    run_slot("wake")
else:
    print("skipped")


In [ ]:
# === Train slot: confirm ("yes") ===
if "confirm" in TRAIN_SLOTS:
    run_slot("confirm")
else:
    print("skipped")


In [ ]:
# === Package + download ===
import json, os, zipfile

files = []
for slot in TRAIN_SLOTS:
    name = SLOTS[slot]['output_name']
    if os.path.exists(f'{name}.tflite'):
        files.append(f'{name}.tflite')
        meta = {'type': 'micro', 'wake_word': SLOTS[slot]['wake_word'],
                'model': f'{name}.tflite', 'trained_languages': ['en'],
                'version': 2,
                'micro': {'probability_cutoff': 0.97 if slot == 'wake' else 0.85,
                          'feature_step_size': 10, 'sliding_window_size': 5,
                          'tensor_arena_size': 50000}}
        with open(f'{name}.json', 'w') as f:
            json.dump(meta, f, indent=2)
        files.append(f'{name}.json')

assert files, 'nothing trained?'
with zipfile.ZipFile('pip_voice_models.zip', 'w') as z:
    for f in files:
        z.write(f)
print('zipped:', files)
if DRIVE_DIR:
    import shutil
    shutil.copy2('pip_voice_models.zip', DRIVE_DIR)
    print('copied to', DRIVE_DIR)

from google.colab import files as colab_files
colab_files.download('pip_voice_models.zip')


## Back on the Mac

```bash
cd ~/pip/tools/wakeword
unzip ~/Downloads/pip_voice_models.zip -d models/
./install_models.sh          # regenerates the firmware C arrays + rebuilds
```

Before embedding, sanity-check with real audio (see README):
`python validate.py models/hey_pip.tflite positives/ negatives/` — the
cutoff sweep it prints is what you pass to `install_models.sh` if the
defaults (0.97 wake / 0.85 confirm) look wrong. On-device symptoms map the
same way: doesn't fire → lower the cutoff and re-run `install_models.sh`;
fires on TV/chatter → raise it.

**Optional personal fine-tune** (if the kid's voice trips detection): record
him saying each phrase 50–100× on a Pip box, decode the `.vmsg`s to wavs
server-side (`app/vmsg.py`), upload them to `real_recordings_wake/` /
`real_recordings_confirm/` in the Colab working dir (or Drive, then copy in),
and re-run — the pipeline picks the folders up automatically as
heavily-weighted positive sets.

## Stopping early / recovering a model from checkpoints

Exports always use `best_weights` (the checkpoint with the fewest ambient
false-fires so far), so a run deep into training can be cut without losing
much — the first `hey_pip.tflite` was exported this way at step 28k/45k.
Interrupt the cell (Runtime → Interrupt execution), then run in a new cell
(swap `wake`/`hey_pip` for `confirm`/`yes` as needed):

```python
import os, shutil, subprocess, sys
subprocess.run(['pkill', '-f', 'model_train_eval'], check=False)
env = os.environ.copy()
env['PYTHONPATH'] = '/content/microWakeWord:' + env.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config', 'training_parameters_wake.yaml',
    '--train', '0', '--restore_checkpoint', '1',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet', '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0', '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5', '--stride', '3'],
    env=env, check=True)
shutil.copy2('trained_models/hey_pip/tflite_stream_state_internal_quant/'
             'stream_state_internal_quant.tflite', 'hey_pip.tflite')
if DRIVE_DIR: shutil.copy2('hey_pip.tflite', DRIVE_DIR)
from google.colab import files; files.download('hey_pip.tflite')
```
